# Clase 071 — Regresión polinomial

Ajustamos modelos **lineales en los coeficientes** a relaciones no lineales usando `PolynomialFeatures`, visualizamos el efecto del grado, diagnosticamos overfitting con train/test y verificamos la explosión combinatoria de features.

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

## 1. Dataset cuadrático ruidoso

$y = 0.5x^2 + x + 2 + \text{ruido}$, con $x\in[-3, 3]$ y $n=100$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
m = 100
X = 6 * np.random.rand(m, 1) - 3
y = 0.5 * X[:, 0]**2 + X[:, 0] + 2 + np.random.randn(m)   # DGP: 0.5x²+x+2+ruido

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(X, y, s=15, alpha=0.7)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_title('Dataset cuadrático ruidoso')
plt.tight_layout(); plt.show()
print('X', X.shape, '| y', y.shape)

## 2. Ajuste de grado 2

`PolynomialFeatures(degree=2, include_bias=False)` genera $[x, x^2]$; `LinearRegression` recupera aproximadamente los coeficientes del DGP: intercept 2, $x$ con 1, $x^2$ con 0.5.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

poly = PolynomialFeatures(degree=2, include_bias=False)
Xp = poly.fit_transform(X)
lin = LinearRegression().fit(Xp, y)
print('intercept_:', round(lin.intercept_, 3), '(DGP = 2)')
print('coef_ [x, x²]:', lin.coef_.round(3), '(DGP = [1, 0.5])')
assert abs(lin.intercept_ - 2) < 0.6 and abs(lin.coef_[1] - 0.5) < 0.3
print('OK: recupera aproximadamente el DGP 0.5x²+x+2')

## 3. Curvas según el grado

Ajustamos grados 1, 2, 5 y 30. El grado 1 subajusta (recta); el grado 30 oscila salvajemente para pasar cerca de cada punto (overfitting).

In [ ]:
from sklearn.pipeline import make_pipeline

X_plot = np.linspace(-3, 3, 200).reshape(-1, 1)
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(X, y, s=12, alpha=0.5, color='gray')
for d, style in zip([1, 2, 5, 30], ['-', '-', '--', ':']):
    model = make_pipeline(PolynomialFeatures(d, include_bias=False), LinearRegression()).fit(X, y)
    ax.plot(X_plot, model.predict(X_plot), style, label=f'grado {d}')
ax.set_ylim(y.min() - 2, y.max() + 2); ax.legend(); ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Ajuste polinómico según el grado')
plt.tight_layout(); plt.show()

## 4. RMSE train vs test según el grado

Con un split 70/30 barremos grados 1 a 15. La curva de train baja monótona; la de test tiene forma de **U**: sube cuando el modelo empieza a sobreajustar.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)
degrees = list(range(1, 16))
rmse_tr, rmse_te = [], []
for d in degrees:
    model = make_pipeline(PolynomialFeatures(d, include_bias=False), LinearRegression()).fit(Xtr, ytr)
    rmse_tr.append(np.sqrt(mean_squared_error(ytr, model.predict(Xtr))))
    rmse_te.append(np.sqrt(mean_squared_error(yte, model.predict(Xte))))
best = degrees[int(np.argmin(rmse_te))]
print('grado óptimo (min RMSE test):', best)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(degrees, rmse_tr, 'o-', label='train')
ax.plot(degrees, rmse_te, 's-', label='test')
ax.axvline(best, color='k', ls=':'); ax.legend()
ax.set_xlabel('grado'); ax.set_ylabel('RMSE'); ax.set_title('Overfitting: la RMSE de test tiene forma de U')
plt.tight_layout(); plt.show()
assert best <= 6

## 5. Explosión combinatoria

Con $n$ features de entrada y grado $d$, la cantidad de columnas generadas (sin bias) es $C(n+d, d) - 1$. Lo verificamos con $n=3$, $d=4$.

In [ ]:
from math import comb

poly = PolynomialFeatures(degree=4, include_bias=False)
n_out = poly.fit_transform(np.zeros((1, 3))).shape[1]
esperado = comb(3 + 4, 4) - 1
print('features generadas:', n_out, '| fórmula C(n+d, d) - 1:', esperado)
assert n_out == esperado
print('OK: la combinatoria coincide con C(n+d, d) - 1')

## Ejercicios

1. **Dataset seno.** Generá $y = \sin(x) + \text{ruido}$ en $x\in[0, 2\pi]$, barré grados 1 a 15 con split 80/20 y reportá el grado óptimo según test (debería caer entre 3 y 7).
2. **interaction_only.** Con dos features, compará `PolynomialFeatures(degree=2)` vs `interaction_only=True`: ¿qué columnas desaparecen?
3. **Escalar antes de expandir.** Multiplicá `x` por 1000 y ajustá grado 5 con y sin `StandardScaler` previo. Observá cómo se degradan los coeficientes sin escalado.
4. **Pipeline correcto.** Reescribí el barrido usando `Pipeline` para evitar el error clásico de `fit_transform` sobre el test set.

## Conclusiones

- La regresión polinomial es un modelo **lineal en los coeficientes** sobre features transformadas: por eso basta `LinearRegression`.
- Subir el grado siempre mejora el train, pero el test empeora a partir de cierto punto: es el síntoma canónico del overfitting.
- La cantidad de features crece como $C(n+d, d)$, así que grados altos con muchas features son inviables en memoria.
- Escalar con `StandardScaler` antes de `PolynomialFeatures` evita el mal condicionamiento numérico de las potencias.